# Forecasting

**Perishable Demand Forecasting & Zero-Waste Inventory Engine** - stage 2 of 4.

Notebook 01 filled in the demand that stockouts hid. This notebook forecasts it.

| Step | What it does |
|---|---|
| 1 | load the recovered demand from notebook 01 |
| 2 | search hyperparameters, then train the best one |
| 3 | compare against the baselines |
| 4 | train the same model on raw sales and compare the two |

A **range**, not a number: the model outputs `q10`/`q50`/`q90`, because ordering perishables means
knowing the downside as well as the middle. The test week is never touched here.

Runs on CPU, but the GPU runtime is far faster (Colab: *Runtime -> Change runtime type -> GPU*).

## Setup

In [2]:
import os
os.chdir('/Users/Kavitha/Documents/retail-demand-forecasting-main')

In [3]:
import warnings

import pandas as pd
import torch
warnings.filterwarnings("ignore")   # pytorch-forecasting is noisy about dataloader workers

from src.utils import config, data_io
from src.utils.metrics import quantile_scores
from src import forecast, recovery

if torch.cuda.is_available():
    gpu = torch.cuda.get_device_name(0)
elif torch.backends.mps.is_available():
    gpu = "Apple GPU (MPS)"
else:
    gpu = "none (CPU)"
print(f"torch {torch.__version__} | GPU: {gpu}")

torch 2.13.0 | GPU: Apple GPU (MPS)


## 1. Load the recovered demand

In [4]:
daily = recovery.load_daily().merge(
    data_io.load("recovered")[["store_id", "product_id", "dt", "recovered_demand"]],
    on=["store_id", "product_id", "dt"], how="left")

assert daily["recovered_demand"].notna().all(), "run notebook 01 step 4 first"
print(f"{len(daily):,} rows | recovery model:", recovery.load_params()["model"])
daily.groupby("period")[["sale_amount", "recovered_demand"]].mean().round(4)

loading daily subset from /Users/Kavitha/Documents/retail-demand-forecasting-main/data/processed/daily_train.parquet + /Users/Kavitha/Documents/retail-demand-forecasting-main/data/processed/daily_eval.parquet
loading recovered subset from /Users/Kavitha/Documents/retail-demand-forecasting-main/data/processed/daily_train_recovered.parquet + /Users/Kavitha/Documents/retail-demand-forecasting-main/data/processed/daily_eval_recovered.parquet
543,297 rows | recovery model: lightgbm_tweedie


,sale_amount,recovered_demand
period,,
calibration,1.1303,1.2861
test,1.1990,1.3506
training,0.9555,1.1457
validation,1.0471,1.1857


## 2. Hyperparameter search

`forecast.GRID` defines the configurations searched. Every setting in it is fitted and scored on the
**validation** window; anything not in it is held at its default in `forecast.train`.

| setting | what it controls |
|---|---|
| `learning_rate` | step size. Too high and it never settles; too low and it stops before learning |
| `encoder_days` | history read per forecast. Longer sees more seasonality but gives fewer training windows |
| `hidden_size` | model capacity. More data supports more of it |
| `dropout` | regularisation strength |

An earlier full-factorial search on a smaller subset found that **no setting moved validation pinball
by more than the seed-to-seed spread**. That result is why this grid is small: settings whose effect
is buried in noise are held fixed at the level that was best *averaged over all the others*, which
averages the noise down instead of chasing one lucky run. Only settings with a specific reason to be
re-tested on this larger subset are searched.

**Scored on `pinball(avg)`**, not WAPE: the ordering stage consumes the whole q10/q50/q90 range, so
the metric that scores the range should pick the model.

Configs are ranked on the **mean over seeds**, with `pinball_spread` beside it. A gap between configs
smaller than that spread is not a result.

In [5]:
TUNE = True   # False = load the saved ranking; re-run after a disconnect to resume

tuning = (forecast.tune(daily, max_epochs=15) if TUNE
          else pd.read_csv(config.tft_tuning("recovered")))
tuning.drop(columns="config_id").round(4)


=== config 1/24: {'learning_rate': 0.005, 'dropout': 0.2, 'encoder_days': 3, 'weight_decay': 0.001} x 1 seed(s) ===
[recovered] Apple GPU (MPS) | batch=1024 workers=4 seed=123
  epoch  0  train_loss=0.1666  val_loss=0.2031
  epoch  1  train_loss=0.1579  val_loss=0.1995
  epoch  2  train_loss=0.1533  val_loss=0.1914
  epoch  3  train_loss=0.1516  val_loss=0.1903
  epoch  4  train_loss=0.1532  val_loss=0.1932
  epoch  5  train_loss=0.1542  val_loss=0.1921
  epoch  6  train_loss=0.1552  val_loss=0.1972
  epoch  7  train_loss=0.1551  val_loss=0.1932
  epoch  8  train_loss=0.1549  val_loss=0.1949
  epoch  9  train_loss=0.1560  val_loss=0.1943
  epoch 10  train_loss=0.1549  val_loss=0.1989
  epoch 11  train_loss=0.1549  val_loss=0.1933
    -> pinball(avg)=0.123 (spread 0.0)  WAPE=0.3756

=== config 2/24: {'learning_rate': 0.005, 'dropout': 0.2, 'encoder_days': 3, 'weight_decay': 0.0001} x 1 seed(s) ===
[recovered] Apple GPU (MPS) | batch=1024 workers=4 seed=123
  epoch  0  train_loss=0.1575

,learning_rate,dropout,encoder_days,weight_decay,WAPE,WPE,MAE,pinball(avg),CRPS~,pinball_spread,n_seeds
0,0.010,0.3,3,0.0001,0.3151,0.0405,0.3220,0.1031,0.2006,0.0,1
1,0.010,0.3,7,0.0001,0.3212,0.0788,0.3290,0.1055,0.2054,0.0,1
2,0.005,0.3,3,0.0001,0.3212,0.0835,0.3289,0.1055,0.2054,0.0,1
3,0.010,0.2,7,0.0001,0.3237,0.0920,0.3312,0.1063,0.2068,0.0,1
4,0.010,0.2,5,0.0001,0.3258,0.0927,0.3338,0.1063,0.2069,0.0,1
5,0.010,0.2,3,0.0001,0.3289,0.0743,0.3367,0.1071,0.2085,0.0,1
6,0.005,0.3,7,0.0001,0.3253,0.0991,0.3341,0.1072,0.2087,0.0,1
7,0.005,0.2,3,0.0001,0.3296,0.1079,0.3384,0.1072,0.2086,0.0,1
8,0.005,0.3,5,0.0001,0.3279,0.0905,0.3365,0.1077,0.2097,0.0,1
9,0.005,0.2,5,0.0001,0.3318,0.1224,0.3402,0.1099,0.2140,0.0,1


### Train the best settings

The winner is refitted with the same epoch ceiling as the search, so the model that is saved is the
one the ranking actually measured.

| | |
|---|---|
| **Target** | `recovered_demand` (raw `sale_amount` in step 4) |
| **History** | the winning `encoder_days`, read by the model itself - no hand-built lag columns |
| **Known ahead** | day of week, discount, holiday, activity, weather |
| **Per product** | store, product and the three category IDs, as learned embeddings |
| **Horizon** | 7 days, rolled forward across the period |

Early stopping watches `val_loss` and the best epoch is checkpointed and reloaded, so the saved model
is the best one seen rather than the last one trained. Each 7-day block is forecast from history that
stops the day before it starts, so no block sees inside itself.

In [6]:
BEST = forecast.best_params(tuning)
print("winning config:", BEST)

TRAIN   = True
PERIODS = ("validation", "calibration")

if TRAIN:
    recovered = forecast.run(daily, tag="recovered", max_epochs=15, **BEST)
else:
    recovered = {p: pd.read_parquet(config.forecast_parquet(p, "recovered")) for p in PERIODS}

recovered["validation"][["dt", "store_id", "product_id",
                         "q10", "q50", "q90", "sale_amount", "is_censored"]].head()

winning config: {'learning_rate': 0.01, 'dropout': 0.3, 'encoder_days': 3, 'weight_decay': 0.0001}
[recovered] Apple GPU (MPS) | batch=1024 workers=4 seed=123
  epoch  0  train_loss=0.1568  val_loss=0.1867
  epoch  1  train_loss=0.1439  val_loss=0.1734
  epoch  2  train_loss=0.1412  val_loss=0.1786
  epoch  3  train_loss=0.1403  val_loss=0.1744
  epoch  4  train_loss=0.1403  val_loss=0.1776
  epoch  5  train_loss=0.1395  val_loss=0.1770
  epoch  6  train_loss=0.1395  val_loss=0.1758
  epoch  7  train_loss=0.1391  val_loss=0.1803
  epoch  8  train_loss=0.1391  val_loss=0.1863
  epoch  9  train_loss=0.1391  val_loss=0.1792
[recovered] validation: {'WAPE': np.float64(0.3284), 'WPE': np.float64(0.0851), 'MAE': np.float64(0.3358), 'pinball(avg)': 0.1074, 'CRPS~': np.float64(0.2091)}


,dt,store_id,product_id,q10,q50,q90,sale_amount,is_censored
0,2024-05-29,10,116,0.326862,0.601307,0.945230,0.9,1
1,2024-05-30,10,116,0.289122,0.550583,0.876019,0.9,0
2,2024-05-31,10,116,0.328808,0.604475,0.943175,0.4,0
3,2024-06-01,10,116,0.438102,0.756338,1.149172,0.6,0
4,2024-06-02,10,116,0.488201,0.824321,1.235867,0.2,0


## 3. Compare against the baselines

Same scoring as notebook 01: **per date, on non-stockout rows only**, against recorded
`sale_amount` - which keeps these numbers comparable to the baseline scorecard.

- **WAPE / MAE** - how far off the middle guess (`q50`) is. Lower is better.
- **WPE** - direction. Positive over-forecasts, negative under-forecasts.
- **pinball(avg) / CRPS~** - whether the whole range is right, not just the middle. This is what the
  ordering stage actually uses.

In [7]:
baselines = pd.read_csv(config.BASELINE_SCORECARD, index_col=0)

pd.concat([pd.DataFrame({"tft_recovered": quantile_scores(recovered["validation"])}).T,
           baselines]).round(4)

,WAPE,WPE,MAE,pinball(avg),CRPS~,pinball@0.5,n_series,n_failed
tft_recovered,0.3284,0.0851,0.3358,0.1074,0.2091,NaN,NaN,NaN
seasonal_naive,0.4213,0.0187,0.4301,NaN,NaN,0.2212,NaN,NaN
xgboost_quantile,0.3423,-0.0187,0.3502,0.1121,0.2181,NaN,NaN,NaN
sarima,0.5210,-0.2521,0.4149,NaN,NaN,NaN,30.0,0.0


## 4. Recovered demand vs raw sales

The same settings, seed and features, trained on raw `sale_amount` instead of `recovered_demand`.
Only the target changes.

**Pooled, this table cannot settle which is better** - and it will make recovery look slightly
worse. Scoring skips stockout rows, and stockout rows are the only rows recovery changes, so on the
rows being scored the two targets are *identical*. The raw version predicts exactly what it is
measured against; the recovered version predicts demand and is marked down for exceeding recorded
sales.

The question that *can* be settled is asked in **notebook 01 section 6**, which runs this same
comparison on the XGBoost baseline and splits the score by how often each product sells out. There,
recovery removes the under-forecast on the products that sell out most, at the cost of a mild
over-forecast on everything else. Expect the TFT to behave the same way; the pooled row here is the
cost side of that trade, not a verdict.

In [8]:
# the SAME winning settings: only the target changes
if TRAIN:
    raw = forecast.run(daily, tag="raw", max_epochs=15, **BEST)
else:
    raw = {p: pd.read_parquet(config.forecast_parquet(p, "raw")) for p in PERIODS}

both = {"tft_recovered": recovered["validation"], "tft_raw": raw["validation"]}
scorecard = pd.concat([pd.DataFrame({k: quantile_scores(v) for k, v in both.items()}).T, baselines])
scorecard.to_csv(config.FORECAST_SCORECARD)
scorecard.round(4)

[raw] Apple GPU (MPS) | batch=1024 workers=4 seed=123
  epoch  0  train_loss=0.1675  val_loss=0.2017
  epoch  1  train_loss=0.1567  val_loss=0.1888
  epoch  2  train_loss=0.1543  val_loss=0.1879
  epoch  3  train_loss=0.1534  val_loss=0.1890
  epoch  4  train_loss=0.1529  val_loss=0.1905
  epoch  5  train_loss=0.1526  val_loss=0.1841
  epoch  6  train_loss=0.1524  val_loss=0.1868
  epoch  7  train_loss=0.1520  val_loss=0.1869
  epoch  8  train_loss=0.1520  val_loss=0.1872
  epoch  9  train_loss=0.1518  val_loss=0.1874
  epoch 10  train_loss=0.1515  val_loss=0.1834
  epoch 11  train_loss=0.1517  val_loss=0.1868
  epoch 12  train_loss=0.1514  val_loss=0.1836
  epoch 13  train_loss=0.1514  val_loss=0.1830
  epoch 14  train_loss=0.1514  val_loss=0.1845
[raw] validation: {'WAPE': np.float64(0.3286), 'WPE': np.float64(-0.0656), 'MAE': np.float64(0.3364), 'pinball(avg)': 0.1089, 'CRPS~': np.float64(0.212)}


,WAPE,WPE,MAE,pinball(avg),CRPS~,pinball@0.5,n_series,n_failed
tft_recovered,0.3284,0.0851,0.3358,0.1074,0.2091,NaN,NaN,NaN
tft_raw,0.3286,-0.0656,0.3364,0.1089,0.2120,NaN,NaN,NaN
seasonal_naive,0.4213,0.0187,0.4301,NaN,NaN,0.2212,NaN,NaN
xgboost_quantile,0.3423,-0.0187,0.3502,0.1121,0.2181,NaN,NaN,NaN
sarima,0.5210,-0.2521,0.4149,NaN,NaN,NaN,30.0,0.0


### Forecast gap on stockout days

The scoring above hides the difference; the forecasts themselves show it. On days that ran out, the
recovered version should predict noticeably more than the raw one - and on days that did not, the two
should almost agree. That gap is the demand the raw model never learned existed.

In [9]:
keys = ["store_id", "product_id", "dt"]
gap = (recovered["validation"][keys + ["is_censored", "q50"]]
       .rename(columns={"q50": "q50_recovered"})
       .merge(raw["validation"][keys + ["q50"]].rename(columns={"q50": "q50_raw"}), on=keys))

(gap.groupby("is_censored")[["q50_raw", "q50_recovered"]].mean()
    .assign(gap_pct=lambda d: (d.q50_recovered / d.q50_raw - 1) * 100).round(4))

,q50_raw,q50_recovered,gap_pct
is_censored,,,
0,0.9618,1.1151,15.937500
1,1.0236,1.2180,18.992901


## 5. Why the pooled scorecard cannot see recovery

Section 4 shows the two targets nearly level, and that is not a null result - it is the metric
declining to answer. Accuracy is scored on **full-shelf days only**, because those are the only days
where recorded sales are the true demand.

But full-shelf days are not a fair sample of days. **A shelf stays full on a quiet day and empties on
a busy one.** So grading only on full-shelf days means grading on the quiet ones, and a model that
learned the demand level across all days will read high there through no fault of its own.

The cell below measures how high. That figure is the handicap any demand-trained model carries into
section 3 before it makes a single mistake.

In [10]:
from src.utils.features import censoring_bucket
from src.utils.metrics import scores_by_bucket

band = censoring_bucket(daily)   # each series' TRAINING-period stockout rate, cut into four bands


def int_keys(fc):
    """forecast._prepare stores the IDs as STRINGS so the TFT treats them as embeddings, but `band`
    is indexed on the daily frame's integers. Reindexing across that matches nothing and fails
    silently - every band NaN, no groups, a clean-looking table holding only a pooled row."""
    fc = fc.copy()
    fc[["store_id", "product_id"]] = fc[["store_id", "product_id"]].astype(int)
    return fc


tr = daily[daily.period == "training"]
clean, cens = tr[tr.is_censored == 0], tr[tr.is_censored == 1]

print("recovery is a no-op on full-shelf days: max |recovered - recorded| ="
      f" {(clean.recovered_demand - clean.sale_amount).abs().max():.1e}")
print()
print(f"  mean demand, full-shelf days : {clean.recovered_demand.mean():.4f}   <- the days we grade on")
print(f"  mean demand, sold-out days   : {cens.recovered_demand.mean():.4f}")
print(f"  mean demand, all days        : {tr.recovered_demand.mean():.4f}   <- the level a model learns")
print()
handicap = (tr.recovered_demand.mean() / clean.recovered_demand.mean() - 1) * 100
print(f"  => graded on full-shelf days, a demand model reads {handicap:.1f}% high"
      " before any modelling error at all")

keys = ["store_id", "product_id"]
tr = tr.assign(band=band.reindex(pd.MultiIndex.from_frame(tr[keys])).to_numpy())
rows = []
for b, g in tr.groupby("band", observed=True):
    c = g[g.is_censored == 0]
    rows.append({"band": str(b), "clean_days": len(c),
                 "soldout_days": int((g.is_censored == 1).sum()),
                 "demand_clean": round(c.recovered_demand.mean(), 4),
                 "demand_all": round(g.recovered_demand.mean(), 4),
                 "handicap_%": round((g.recovered_demand.mean()
                                      / c.recovered_demand.mean() - 1) * 100, 1)})
print()
print(pd.DataFrame(rows).to_string(index=False))

recovery is a no-op on full-shelf days: max |recovered - recorded| = 3.6e-15

  mean demand, full-shelf days : 0.9224   <- the days we grade on
  mean demand, sold-out days   : 1.4163
  mean demand, all days        : 1.1457   <- the level a model learns

  => graded on full-shelf days, a demand model reads 24.2% high before any modelling error at all

         band  clean_days  soldout_days  demand_clean  demand_all  handicap_%
       25-50%      126583         77769        0.8162      0.9304        14.0
       50-75%       40671         57041        1.2686      1.4487        14.2
<25% censored       20009          4977        0.8001      0.8632         7.9
        >=75%        3030         17182        1.5234      2.2073        44.9


## 6. Where recovery pays, and where it costs

Section 4 is pooled, and section 5 explains why pooled comes back flat. Split the same forecasts by
**how often each product sells out** and the effect has somewhere to show up.

Notebook 01 ran this cut for the XGBoost baseline: the raw-target twin under-forecast the worst band
by 19.6%, and recovery removed essentially all of it for a 21.5% accuracy gain. The question is
whether the model we actually ship does the same.

**It can come back flat.** If it does, recovery helps a model we are not using, and the ordering stage
has nothing to convert into a saving. No refitting - the forecasts are already in memory.

In [11]:
tbl = pd.concat({tag: scores_by_bucket(int_keys(fc["validation"]), band)
                 for tag, fc in [("raw", raw), ("recovered", recovered)]}, axis=1)
tbl[("gap", "WAPE_%")] = ((tbl[("recovered", "WAPE")] / tbl[("raw", "WAPE")] - 1) * 100).round(2)
print(tbl.round(4).to_string())

                   raw                                              recovered                                                 gap
              n_scored    WAPE     WPE     MAE pinball(avg)   CRPS~  n_scored    WAPE     WPE     MAE pinball(avg)   CRPS~ WAPE_%
<25% censored   3955.0  0.4231 -0.0302  0.3066       0.0992  0.2722    3955.0  0.4207  0.0710  0.3047       0.0979  0.2684  -0.57
25-50%         30345.0  0.3512 -0.0445  0.2880       0.0931  0.2261   30345.0  0.3623  0.1052  0.2966       0.0944  0.2291   3.16
50-75%         12319.0  0.2982 -0.0670  0.4139       0.1333  0.1911   12319.0  0.2989  0.0835  0.4144       0.1331  0.1908   0.23
>=75%           1375.0  0.2595 -0.2017  0.7970       0.2655  0.1725    1375.0  0.1940 -0.0027  0.5877       0.1919  0.1247 -25.24
ALL            47994.0  0.3286 -0.0656  0.3364       0.1089  0.2120   47994.0  0.3284  0.0851  0.3358       0.1074  0.2091  -0.06


## 7. Overstock vs understock - the trade in units

WAPE charges the same for one unit too many as one unit too few. **A bin and an empty shelf are not
the same event**, and an empty shelf also produces another censored zero, which teaches the next model
to order even less. So the accuracy tables cannot express what recovery is for.

This splits the error by direction on days demand is known, ordering at `q50`:

- **understock %** - demand that walked away, as a share of demand. The failure recovery targets.
- **overstock %** - units that would have been binned. The price recovery charges.

Read the worst band first. If recovery is doing its job, understock falls sharply there and overstock
rises by less than it saves - and the cost sweep in the ordering stage decides whether that trade pays.

In [12]:
def stock_errors(fc, label):
    """Units under- and over-ordered at q50, on days recorded sales ARE demand."""
    d = int_keys(fc[fc.stock_hour6_22_cnt == 0])
    d["band"] = band.reindex(pd.MultiIndex.from_frame(d[["store_id", "product_id"]])).to_numpy()
    assert d["band"].notna().any(), "no row matched a band - check the ID dtypes"
    d["under"] = (d.sale_amount - d.q50).clip(lower=0)
    d["over"] = (d.q50 - d.sale_amount).clip(lower=0)
    g = d.groupby("band", observed=True).agg(demand=("sale_amount", "sum"),
                                             under=("under", "sum"), over=("over", "sum"))
    return pd.DataFrame({f"understock_%_{label}": (g.under / g.demand * 100).round(1),
                         f"overstock_%_{label}": (g.over / g.demand * 100).round(1)})


trade = pd.concat([stock_errors(raw["validation"], "raw"),
                   stock_errors(recovered["validation"], "recovered")], axis=1)
trade["understock_saved_pts"] = (trade["understock_%_raw"] - trade["understock_%_recovered"]).round(1)
trade["overstock_added_pts"] = (trade["overstock_%_recovered"] - trade["overstock_%_raw"]).round(1)
print(trade.to_string())
print()
print("net points in recovery's favour at a 1:1 cost ratio:")
print((trade["understock_saved_pts"] - trade["overstock_added_pts"]).round(1).to_string())

               understock_%_raw  overstock_%_raw  understock_%_recovered  overstock_%_recovered  understock_saved_pts  overstock_added_pts
band                                                                                                                                      
25-50%                     19.8             15.4                    12.8                   23.3                   7.0                  7.9
50-75%                     18.1             11.6                    10.8                   19.0                   7.3                  7.4
<25% censored              22.8             19.5                    17.6                   24.4                   5.2                  4.9
>=75%                      22.7              2.8                     9.8                    9.1                  12.9                  6.3

net points in recovery's favour at a 1:1 cost ratio:
band
25-50%          -0.9
50-75%          -0.1
<25% censored    0.3
>=75%            6.6


### How much does the score move on its own?

Every gap above has to be read against this. GPU training is not bit-reproducible, so weight
initialisation and batch order shift between runs of identical settings - and on MPS even the same
seed does not reproduce exactly.

**Both targets, three seeds each.** Running only the recovered arm would give a spread but no way to
compare it against anything: the raw arm has a spread too, and the quantity that matters is the
*gap between them*, seed by seed.

The forecasts are kept in memory rather than saved, so the per-band table from section 6 can be
recomputed for each seed. That turns "recovery improves every band" from a single draw into a mean
with a spread - which is the form the 25-50% band needs, since its gap is currently smaller than the
pooled seed spread.

`save=False` - throwaway fits must not replace the shipped model or forecasts.

In [13]:
SEEDS = (123, 456, 789)

# 3 seeds x 2 targets. Frames retained so the per-band gap can be recomputed per seed.
seed_fc = {(tag, s): forecast.run(daily, tag=tag, seed=s, max_epochs=15,
                                  periods=("validation",), save=False, **BEST)["validation"]
           for tag in ("raw", "recovered") for s in SEEDS}

pooled = pd.DataFrame({f"{tag}_seed{s}": quantile_scores(fc)
                       for (tag, s), fc in seed_fc.items()}).T
for tag in ("raw", "recovered"):
    rows = pooled.loc[[i for i in pooled.index if i.startswith(tag)]]
    pooled.loc[f"{tag}_MEAN"] = rows.mean()
    pooled.loc[f"{tag}_SPREAD"] = rows.max() - rows.min()
print(pooled.round(4).to_string())

# the headline: per-band WAPE gap, recovered vs raw, one column per seed
gaps = {}
for s in SEEDS:
    r = scores_by_bucket(int_keys(seed_fc[("raw", s)]), band)["WAPE"]
    c = scores_by_bucket(int_keys(seed_fc[("recovered", s)]), band)["WAPE"]
    gaps[f"seed{s}"] = ((c / r - 1) * 100).astype(float).round(2)

g = pd.DataFrame(gaps)
g["MEAN"] = g.mean(axis=1).round(2)
g["SPREAD"] = (g.max(axis=1) - g.min(axis=1)).round(2)
print()
print("per-band WAPE gap %, recovered vs raw (negative = recovery better):")
print(g.to_string())
print()
print("a band whose |MEAN| is smaller than its SPREAD is not established.")

[raw] Apple GPU (MPS) | batch=1024 workers=4 seed=123
  epoch  0  train_loss=0.1675  val_loss=0.1953
  epoch  1  train_loss=0.1562  val_loss=0.1846
  epoch  2  train_loss=0.1542  val_loss=0.1852
  epoch  3  train_loss=0.1533  val_loss=0.1837
  epoch  4  train_loss=0.1527  val_loss=0.1894
  epoch  5  train_loss=0.1524  val_loss=0.1832
  epoch  6  train_loss=0.1520  val_loss=0.1876
  epoch  7  train_loss=0.1518  val_loss=0.1854
  epoch  8  train_loss=0.1517  val_loss=0.1896
  epoch  9  train_loss=0.1516  val_loss=0.1862
  epoch 10  train_loss=0.1514  val_loss=0.1842
  epoch 11  train_loss=0.1515  val_loss=0.1839
  epoch 12  train_loss=0.1513  val_loss=0.1853
  epoch 13  train_loss=0.1513  val_loss=0.1810
  epoch 14  train_loss=0.1513  val_loss=0.1820
[raw] validation: {'WAPE': np.float64(0.3289), 'WPE': np.float64(-0.0462), 'MAE': np.float64(0.3367), 'pinball(avg)': 0.1087, 'CRPS~': np.float64(0.2116)}
[raw] Apple GPU (MPS) | batch=1024 workers=4 seed=456
  epoch  0  train_loss=0.1655  v